In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, numpy as np

def to_native(o):
    if isinstance(o, dict):  return {k: to_native(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_native(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o

payload = {
    'runs': to_native(results),
    'statistic': STAT_NAMES[STAT_IDX],
    'area_only_auc': float(roc_auc_score(Y, -AREA)),
    'shape_only_auc': float(shape_auc),
    'teacher_auc_mean': float(np.mean(ta)), 'teacher_auc_std': float(np.std(ta)),
    'student_auc_mean': float(np.mean(sa)), 'student_auc_std': float(np.std(sa)),
    'pseudo_acc_mean': float(np.mean(pa)),
    'n_images': int(len(Y)),
}

path = os.path.join(RESULTS_DIR, 'corrected_pipeline.json')
tmp = path + '.tmp'
with open(tmp, 'w') as f:
    json.dump(payload, f, indent=2, default=to_native)
os.replace(tmp, path)   # atomic: no more half-written files

print("saved:", path)
print(json.dumps(payload, indent=2)[:600])

In [ ]:
import os, json, glob

BASE = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"

for folder in ["results", "deploy", "cross_validation_datasets"]:
    d = os.path.join(BASE, folder)
    if not os.path.isdir(d):
        print(f"[skip] {folder} not found"); continue
    print(f"\n{'='*70}\n{folder}/\n{'='*70}")
    for path in sorted(glob.glob(os.path.join(d, "**", "*.json"), recursive=True)):
        print(f"\n--- {os.path.relpath(path, BASE)} ---")
        try:
            with open(path) as f:
                data = json.load(f)
        except Exception as e:
            print(f"  [could not read: {e}]"); continue
        # print top-level structure and any AUC-like numbers
        def walk(obj, prefix=""):
            if isinstance(obj, dict):
                for k, v in obj.items():
                    walk(v, f"{prefix}{k}.")
            elif isinstance(obj, (int, float)):
                print(f"  {prefix[:-1]} = {obj}")
            elif isinstance(obj, list) and len(obj) <= 12:
                print(f"  {prefix[:-1]} = {obj}")
            elif isinstance(obj, list):
                print(f"  {prefix[:-1]} = [{len(obj)} items]")
        walk(data)